# proctor regression
re-analysis of the proctor et al. 2018 maize yield regression: yield left unexplained by their weather controls, and the partial effects of srm with countries weighted by maize area (figure 5)

In [1]:
from pathlib import Path
import numpy as np, pandas as pd

raw, data, out = Path("../data/raw/proctor2018"), Path("data"), Path("results")
out.mkdir(exist_ok=True)
safrica = ["ZAF", "ZWE", "ZMB", "MWI", "MOZ", "BWA", "NAM", "LSO", "SWZ"]

In [2]:
# fao maize yield (element 4) for countries with all 26 years 1980 to 2005, harvested area (element 1), enso index
# by teleconnection region, as in their clean_data.do
f = raw / "panel_Maize.dta"
panel = pd.read_stata(f, convert_categoricals=False)
panel["iso"] = pd.read_stata(f, columns=["iso3c"]).iso3c.astype(str).values
area = panel[panel.ElementName == 1].set_index(["AreaName", "Year"]).Value.rename("area_ha")
p = panel[(panel.ElementName == 4) & (panel.number_obs_1980_2005 == 26)].join(area, on=["AreaName", "Year"])
p["lyield"], p["trend"] = np.log(p.Value), p.Year - 1950
e = pd.read_stata(raw / "Hsiang_Meng_AER_PP_repData.dta", columns=["iso", "year", "nino34_late", "L_nino34_late", "enso_region"])
p = p.merge(e, left_on=["iso", "Year"], right_on=["iso", "year"], how="left")
for r in sorted(p.enso_region.dropna().unique()):
    p[f"nino_x_{r}"] = p.nino34_late * (p.enso_region == r)
    p[f"Lnino_x_{r}"] = p.L_nino34_late * (p.enso_region == r)
prefix = {"T": "tavg_RCS_k4", "P": "UDel_RCS_k3", "C": "PctDCloudy_RCS_total_k3", "E": ("nino_x_", "Lnino_x_")}
blocks = {"A": ["giss_saod"], **{k: [c for c in p.columns if c.startswith(v)] for k, v in prefix.items()}}
p = p.dropna(subset=["lyield", "trend"] + sum(blocks.values(), [])).reset_index(drop=True)
len(p), p.AreaName.nunique()

Out[2]: (2505, 104)


(2505, 104)

In [3]:
def partial_out(d, cols):
    # country fixed effects and country quadratic trends
    res = d[cols].astype(float).copy()
    for i in d.groupby("AreaName").indices.values():
        t = d.trend.values[i].astype(float)
        z = np.column_stack([np.ones_like(t), t, t ** 2])
        b, *_ = np.linalg.lstsq(z, res.values[i], rcond=None)
        res.iloc[i] = res.values[i] - z @ b
    return res


def twoway(x, u, g1, g2):
    # covariance clustered by country and year
    s = x.std(0)
    x = x / s
    xtxi = np.linalg.inv(x.T @ x)

    def meat(g):
        a = pd.DataFrame(x * u[:, None]).groupby(g).sum().values
        return len(a) / (len(a) - 1) * a.T @ a
    v = xtxi @ (meat(g1) + meat(g2) - meat(pd.Series(list(zip(g1, g2))).astype(str).values)) @ xtxi
    w, e = np.linalg.eigh(v)
    return e @ np.diag(np.clip(w, 0, None)) @ e.T / np.outer(s, s)

In [4]:
# yield left after the controls, by region and year
ctrl = sum((blocks[k] for k in "TPCE"), [])
r = partial_out(p, ["lyield"] + ctrl)
c = r[[k for k in ctrl if np.abs(r[k]).sum() > 1e-10]].values
b, *_ = np.linalg.lstsq(c, r.lyield.values, rcond=None)
p["resid"] = r.lyield.values - c @ b
p["group"] = np.where(p.iso.isin(safrica), "Southern Africa", "Rest of world")
resid = p[p.Year.between(1988, 1996)].groupby(["Year", "group"]).resid.mean().unstack()
resid.to_csv(out / "proctor_residuals.csv")
(100 * resid).round(1)

Out[4]: 
group  Rest of world  Southern Africa
Year                                 
1988             2.4              7.4
1989             1.1              4.0
1990             2.0              5.9
1991             3.3             -2.8
1992            -0.8            -51.9
1993            -3.5              7.2
1994            -3.4              0.8
1995            -2.0             12.5
1996            -1.4             13.1


group,Rest of world,Southern Africa
Year,,
1988,2.4,7.4
1989,1.1,4.0
1990,2.0,5.9
1991,3.3,-2.8
1992,-0.8,-51.9
1993,-3.5,7.2
1994,-3.4,0.8
1995,-2.0,12.5
1996,-1.4,13.1


In [5]:
# partial effects of srm, g3 minus rcp4.5 basis terms times the coefficients
terms = {"tas": blocks["T"], "saod": blocks["A"], "clt": blocks["C"], "pr": blocks["P"]}
basis = pd.read_csv(data / "proctor_fig4e_maize_basis.csv")
basis = basis[basis.term != "saved_effect"].pivot_table(index=["var", "term"], columns="scenario", values="value")
delta = {terms[v][int(k)]: x for (v, k), x in (basis.G3 - basis.rcp45).items()}
cols = sum((blocks[k] for k in "ATPCE"), [])

rows = []
for version, weighted, drop in [("Proctor et al. (2018)", False, False), ("Area-weighted", True, False),
                                ("Area-weighted, southern Africa excluded", True, True)]:
    d = p[~p.iso.isin(safrica)].reset_index(drop=True) if drop else p
    r = partial_out(d, ["lyield"] + cols)
    keep = [k for k in cols if np.abs(r[k].values).sum() > 1e-10]
    w = d.groupby("AreaName").area_ha.transform("mean").values if weighted else np.ones(len(d))
    sw = np.sqrt(w / w.mean())
    x, y = r[keep].values * sw[:, None], r.lyield.values * sw
    b, *_ = np.linalg.lstsq(x, y, rcond=None)
    v = pd.DataFrame(twoway(x, y - x @ b, d.AreaName.values, d.Year.values), keep, keep)
    b = pd.Series(b, keep)
    for effect, names in list(terms.items()) + [("total", sum(terms.values(), []))]:
        dv = np.array([delta[n] for n in names])
        rows.append({"version": version, "effect": effect, "value": 100 * dv @ b[names].values,
                     "ci95": 100 * 1.96 * np.sqrt(dv @ v.loc[names, names].values @ dv)})
effects = pd.DataFrame(rows)
effects.to_csv(out / "proctor_effects.csv", index=False)
effects.pivot(index="effect", columns="version", values="value").round(2)

Out[5]: 
version  Area-weighted  ...  Proctor et al. (2018)
effect                  ...                       
clt               0.01  ...                   0.11
pr                0.33  ...                   0.16
saod             -3.05  ...                  -5.48
tas               5.05  ...                   6.10
total             2.34  ...                   0.88

[5 rows x 3 columns]


version,Area-weighted,"Area-weighted, southern Africa excluded",Proctor et al. (2018)
effect,,,
clt,0.01,-0.17,0.11
pr,0.33,0.29,0.16
saod,-3.05,0.05,-5.48
tas,5.05,4.11,6.10
total,2.34,4.28,0.88
